In [16]:
import pandas as pd

# Day 4 — Step 1.1
# Synthetic Security Master (not real market data)

security_master = pd.DataFrame([
    {
        "security_id": 1001,
        "ticker": "ALPHA",
        "listing_date": "2010-01-04",
        "delisting_date": None,
        "exchange": "NYSE",
        "share_type": "COMMON",
    },
    {
        "security_id": 1002,
        "ticker": "BETA",
        "listing_date": "2011-06-01",
        "delisting_date": "2018-09-28",
        "exchange": "NASDAQ",
        "share_type": "COMMON",
    },
    {
        "security_id": 1003,
        "ticker": "GAMMA",
        "listing_date": "2020-04-15",
        "delisting_date": None,
        "exchange": "NASDAQ",
        "share_type": "COMMON",
    },
    {
        "security_id": 1004,
        "ticker": "DELTA",
        "listing_date": "2012-03-01",
        "delisting_date": None,
        "exchange": "NYSE",
        "share_type": "ETF",
    },
    {
        "security_id": 1005,
        "ticker": "EPSILON",
        "listing_date": "2013-01-02",
        "delisting_date": None,
        "exchange": "OTC",
        "share_type": "COMMON",
    },
])

for column in ["listing_date", "delisting_date"]:
    security_master[column] = pd.to_datetime(
        security_master[column]
    )

display(security_master)

print("Shape:", security_master.shape)
print(
    "Duplicate security IDs:",
    security_master["security_id"].duplicated().sum()
)

,security_id,ticker,listing_date,delisting_date,exchange,share_type
0,1001,ALPHA,2010-01-04,NaT,NYSE,COMMON
1,1002,BETA,2011-06-01,2018-09-28,NASDAQ,COMMON
2,1003,GAMMA,2020-04-15,NaT,NASDAQ,COMMON
3,1004,DELTA,2012-03-01,NaT,NYSE,ETF
4,1005,EPSILON,2013-01-02,NaT,OTC,COMMON


Shape: (5, 6)
Duplicate security IDs: 0


In [17]:
# Day 4 — Step 2.1
# Point-in-Time Universe Eligibility Prototype

def get_pit_universe(security_master, as_of_date):
    as_of_date = pd.Timestamp(as_of_date)

    # 1. Already listed by the selection date
    listed = (
        security_master["listing_date"] <= as_of_date
    )

    # 2. Not delisted before the selection date
    active = (
        security_master["delisting_date"].isna()
        | (security_master["delisting_date"] >= as_of_date)
    )

    # 3. Eligible US exchanges
    valid_exchange = security_master["exchange"].isin(
        ["NYSE", "NASDAQ", "NYSE AMERICAN"]
    )

    # 4. Common shares only
    valid_share_type = (
        security_master["share_type"] == "COMMON"
    )

    eligible = (
        listed
        & active
        & valid_exchange
        & valid_share_type
    )

    return (
        security_master.loc[
            eligible,
            ["security_id", "ticker",
             "listing_date", "delisting_date"]
        ]
        .sort_values("security_id")
        .reset_index(drop=True)
    )


# Test historical universe construction
for date in ["2015-06-30", "2019-06-30", "2021-06-30"]:
    universe = get_pit_universe(
        security_master, date
    )

    print(f"\nUniverse as of {date}:")
    print(universe["ticker"].tolist())


Universe as of 2015-06-30:
['ALPHA', 'BETA']

Universe as of 2019-06-30:
['ALPHA']

Universe as of 2021-06-30:
['ALPHA', 'GAMMA']


In [18]:
# Day 4 — Step 2.2
# Boundary-date checks

test_dates = [
    "2011-05-31",  # Before BETA's listing
    "2011-06-01",  # BETA listing date
    "2018-09-28",  # BETA delisting date
    "2018-10-01",  # After BETA's delisting
    "2020-04-15",  # GAMMA listing date
]

for date in test_dates:
    universe = get_pit_universe(
        security_master, date
    )
    print(
        date,
        "->",
        universe["ticker"].tolist()
    )


2011-05-31 -> ['ALPHA']
2011-06-01 -> ['ALPHA', 'BETA']
2018-09-28 -> ['ALPHA', 'BETA']
2018-10-01 -> ['ALPHA']
2020-04-15 -> ['ALPHA', 'GAMMA']


In [19]:
# Day 4 — Step 3.1
# Automated PIT Universe Tests

def get_tickers(as_of_date):
    universe = get_pit_universe(
        security_master,
        as_of_date
    )
    return set(universe["ticker"])


# Test 1: Historical eligibility
assert get_tickers("2015-06-30") == {
    "ALPHA", "BETA"
}

# Test 2: Delisted security remains
# in the historical universe
assert "BETA" in get_tickers("2015-06-30")

# Test 3: Future IPO must not appear
assert "GAMMA" not in get_tickers("2015-06-30")

# Test 4: ETF must be excluded
assert "DELTA" not in get_tickers("2021-06-30")

# Test 5: OTC security must be excluded
assert "EPSILON" not in get_tickers("2021-06-30")

# Test 6: Listing-date boundary
assert "BETA" not in get_tickers("2011-05-31")
assert "BETA" in get_tickers("2011-06-01")

# Test 7: Delisting-date boundary
assert "BETA" in get_tickers("2018-09-28")
assert "BETA" not in get_tickers("2018-10-01")

print("All 7 PIT universe tests passed.")

All 7 PIT universe tests passed.


In [20]:
# Day 4 — Step 3.2
# Deliberately introduce survivorship bias

# Incorrect approach:
# Remove every security with a delisting date
survivors_only = security_master.loc[
    security_master["delisting_date"].isna()
].copy()

correct_universe = get_pit_universe(
    security_master, "2015-06-30"
)

biased_universe = get_pit_universe(
    survivors_only, "2015-06-30"
)

print(
    "Correct 2015 universe:",
    correct_universe["ticker"].tolist()
)

print(
    "Biased 2015 universe:",
    biased_universe["ticker"].tolist()
)

missing = (
    set(correct_universe["ticker"])
    - set(biased_universe["ticker"])
)

print("Historical securities lost:", missing)

assert "BETA" in missing

print("Survivorship-bias test demonstrated.")

Correct 2015 universe: ['ALPHA', 'BETA']
Biased 2015 universe: ['ALPHA']
Historical securities lost: {'BETA'}
Survivorship-bias test demonstrated.


In [21]:
# Day 5 — Step 1.1
# Synthetic Daily Price and Volume Data

import pandas as pd

# Use business days as a simplified synthetic calendar.
# This is NOT a complete NYSE trading calendar.
dates = pd.bdate_range(
    start="2021-01-04",
    periods=80
)

# Synthetic security IDs from Day 4:
# 1001 = ALPHA
# 1002 = BETA (delisted in 2018)
# 1003 = GAMMA
# 1004 = DELTA (ETF)
# 1005 = EPSILON (OTC)

synthetic_specs = {
    1001: {"close": 100.0, "volume": 200_000},
    1002: {"close": 50.0,  "volume": 500_000},
    1003: {"close": 4.0,   "volume": 5_000_000},
    1004: {"close": 80.0,  "volume": 300_000},
    1005: {"close": 20.0,  "volume": 800_000},
}

records = []

for date in dates:
    for security_id, spec in synthetic_specs.items():
        records.append({
            "date": date,
            "security_id": security_id,
            "close": spec["close"],
            "volume": spec["volume"],
        })

daily_prices = pd.DataFrame(records)

daily_prices["dollar_volume"] = (
    daily_prices["close"] * daily_prices["volume"]
)

daily_prices = daily_prices.sort_values(
    ["security_id", "date"]
).reset_index(drop=True)

display(daily_prices.head(10))

print("Shape:", daily_prices.shape)
print(
    "Date range:",
    daily_prices["date"].min().date(),
    "to",
    daily_prices["date"].max().date()
)

print(
    "Duplicate security-date pairs:",
    daily_prices.duplicated(
        ["security_id", "date"]
    ).sum()
)

,date,security_id,close,volume,dollar_volume
0,2021-01-04,1001,100.0,200000,20000000.0
1,2021-01-05,1001,100.0,200000,20000000.0
2,2021-01-06,1001,100.0,200000,20000000.0
3,2021-01-07,1001,100.0,200000,20000000.0
4,2021-01-08,1001,100.0,200000,20000000.0
5,2021-01-11,1001,100.0,200000,20000000.0
6,2021-01-12,1001,100.0,200000,20000000.0
7,2021-01-13,1001,100.0,200000,20000000.0
8,2021-01-14,1001,100.0,200000,20000000.0
9,2021-01-15,1001,100.0,200000,20000000.0


Shape: (400, 5)
Date range: 2021-01-04 to 2021-04-23
Duplicate security-date pairs: 0


In [22]:
# Day 5 — Step 2.1
# Point-in-Time Price Eligibility

def get_pit_price_universe(
    security_master,
    daily_prices,
    as_of_date,
    min_price=5.0
):
    as_of_date = pd.Timestamp(as_of_date)

    # Step 1: Historical security eligibility
    universe = get_pit_universe(
        security_master,
        as_of_date
    )

    # Step 2: Only use price records at or before cutoff
    historical_prices = daily_prices.loc[
        daily_prices["date"] <= as_of_date
    ].copy()

    # Step 3: Find the latest available price
    # for each security by the cutoff
    latest_prices = (
        historical_prices
        .sort_values(["security_id", "date"])
        .drop_duplicates(
            subset="security_id",
            keep="last"
        )
    )

    # Step 4: Attach price information to the PIT universe
    universe = universe.merge(
        latest_prices[
            ["security_id", "date", "close"]
        ],
        on="security_id",
        how="left",
        validate="one_to_one"
    )

    # Step 5: Apply minimum price threshold
    universe["price_eligible"] = (
        universe["close"].notna()
        & (universe["close"] >= min_price)
    )

    return universe


# Test on a date covered by our synthetic data
result = get_pit_price_universe(
    security_master,
    daily_prices,
    as_of_date="2021-04-15",
    min_price=5.0
)

display(result)


,security_id,ticker,listing_date,delisting_date,date,close,price_eligible
0,1001,ALPHA,2010-01-04,NaT,2021-04-15,100.0,True
1,1003,GAMMA,2020-04-15,NaT,2021-04-15,4.0,False


In [23]:
# Day 5 — Step 2.2
# Automated Price Eligibility Tests

result = get_pit_price_universe(
    security_master,
    daily_prices,
    "2021-04-15"
)

# Test 1: Correct securities after PIT screening
assert set(result["ticker"]) == {
    "ALPHA", "GAMMA"
}

# Test 2: ALPHA passes minimum price
alpha = result.loc[
    result["ticker"] == "ALPHA"
].iloc[0]

assert alpha["close"] == 100.0
assert bool(alpha["price_eligible"])

# Test 3: GAMMA fails minimum price
gamma = result.loc[
    result["ticker"] == "GAMMA"
].iloc[0]

assert gamma["close"] == 4.0
assert not bool(gamma["price_eligible"])

# Test 4: Delisted BETA cannot re-enter
assert "BETA" not in result["ticker"].values

# Test 5: No future price dates were selected
assert (
    result["date"].dropna()
    <= pd.Timestamp("2021-04-15")
).all()

print("All 5 PIT price eligibility tests passed.")

All 5 PIT price eligibility tests passed.


In [24]:
# Day 5 — Step 3.1
# Point-in-Time Median Dollar Volume

def get_pit_liquidity(
    daily_prices,
    as_of_date,
    window=60
):
    as_of_date = pd.Timestamp(as_of_date)

    # Only historical observations
    history = daily_prices.loc[
        daily_prices["date"] <= as_of_date
    ].copy()

    # One row per security per date is required
    if history.duplicated(["security_id", "date"]).any():
        raise ValueError("Duplicate security-date records")

    # Compute dollar volume using compatible units
    history["dollar_volume"] = (
        history["close"] * history["volume"]
    )

    # Sort before taking the trailing window
    history = history.sort_values(
        ["security_id", "date"]
    )

    # Keep the most recent N observations per security
    trailing = (
        history.groupby("security_id")
        .tail(window)
    )

    summary = (
        trailing.groupby("security_id")
        .agg(
            median_dollar_volume=(
                "dollar_volume", "median"
            ),
            observation_count=(
                "dollar_volume", "count"
            )
        )
        .reset_index()
    )

    # Require a complete history window
    summary.loc[
        summary["observation_count"] < window,
        "median_dollar_volume"
    ] = float("nan")

    return summary


liquidity = get_pit_liquidity(
    daily_prices,
    as_of_date="2021-04-15",
    window=60
)

display(liquidity)


,security_id,median_dollar_volume,observation_count
0,1001,20000000.0,60
1,1002,25000000.0,60
2,1003,20000000.0,60
3,1004,24000000.0,60
4,1005,16000000.0,60


In [25]:
# Day 5 — Step 3.2
# Combined Price and Liquidity Eligibility

def get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of_date,
    min_price=5.0,
    min_median_dollar_volume=10_000_000,
    window=60
):
    # Historical security and price eligibility
    universe = get_pit_price_universe(
        security_master,
        daily_prices,
        as_of_date,
        min_price=min_price
    )

    # Historical liquidity summary
    liquidity = get_pit_liquidity(
        daily_prices,
        as_of_date,
        window=window
    )

    # Merge by stable security ID
    universe = universe.merge(
        liquidity,
        on="security_id",
        how="left",
        validate="one_to_one"
    )

    universe["liquidity_eligible"] = (
        universe["median_dollar_volume"].notna()
        & (
            universe["median_dollar_volume"]
            >= min_median_dollar_volume
        )
    )

    universe["eligible"] = (
        universe["price_eligible"]
        & universe["liquidity_eligible"]
    )

    return universe


result = get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of_date="2021-04-15"
)

display(
    result[
        [
            "ticker",
            "close",
            "median_dollar_volume",
            "observation_count",
            "price_eligible",
            "liquidity_eligible",
            "eligible"
        ]
    ]
)


,ticker,close,median_dollar_volume,observation_count,price_eligible,liquidity_eligible,eligible
0,ALPHA,100.0,20000000.0,60,True,True,True
1,GAMMA,4.0,20000000.0,60,False,True,False


In [26]:
# Day 5 — Step 3.3
# Incomplete liquidity window test

early_liquidity = get_pit_liquidity(
    daily_prices,
    as_of_date="2021-01-15",
    window=60
)

display(early_liquidity)

# None of the securities has 60 observations yet
assert early_liquidity[
    "median_dollar_volume"
].isna().all()

print("Incomplete-window test passed.")

,security_id,median_dollar_volume,observation_count
0,1001,NaN,10
1,1002,NaN,10
2,1003,NaN,10
3,1004,NaN,10
4,1005,NaN,10


Incomplete-window test passed.


In [27]:
# Day 5 — Step 4.1
# Automated PIT Price & Liquidity Tests

as_of = "2021-04-15"

result = get_pit_tradable_universe(
    security_master,
    daily_prices,
    as_of
)

# Test 1: Only historically valid securities
assert set(result["ticker"]) == {"ALPHA", "GAMMA"}

# Test 2: Only ALPHA passes both filters
eligible = set(result.loc[
    result["eligible"], "ticker"
])
assert eligible == {"ALPHA"}

# Test 3: Dollar-volume calculations
mdv = result.set_index("ticker")["median_dollar_volume"]
assert mdv["ALPHA"] == 20_000_000
assert mdv["GAMMA"] == 20_000_000

# Test 4: Complete 60-observation windows
assert (result["observation_count"] == 60).all()

# Test 5: Insufficient price history
early = get_pit_tradable_universe(
    security_master,
    daily_prices,
    "2021-01-15"
)
assert not early["eligible"].any()

# Test 6: Future price changes must not affect
# the historical selection result
modified_prices = daily_prices.copy()

future_mask = (
    (modified_prices["security_id"] == 1001)
    & (modified_prices["date"] > pd.Timestamp(as_of))
)

modified_prices.loc[future_mask, "close"] = 1.0

modified_result = get_pit_tradable_universe(
    security_master,
    modified_prices,
    as_of
)

cols = [
    "security_id", "close",
    "median_dollar_volume", "eligible"
]

pd.testing.assert_frame_equal(
    result[cols].reset_index(drop=True),
    modified_result[cols].reset_index(drop=True)
)

# Test 7: Delisted BETA must stay excluded
assert "BETA" not in result["ticker"].values

print("All 7 Day 5 PIT eligibility tests passed.")


All 7 Day 5 PIT eligibility tests passed.
